# Phase Annotation

Complex agents often work in distinct stages — researching, then planning, then executing. **Phase Annotation** provides structured context scoping for these stages using Python's `async with` syntax.

The mechanism is **`self.snapshot`** — it creates a scoped context where think units run, and automatically restores the original state on exit. Think of it as a save/restore point: the agent enters a phase with overridden fields (e.g., a new goal), works within that scope, and when done, everything reverts as if the phase never touched the outer context.

## Initialize

In [1]:
import os

model_name = os.environ.get("MODEL_NAME")
api_key = os.environ.get("API_KEY")
api_base = os.environ.get("BASE_URL")

In [ ]:
from bridgic.llms.openai import OpenAILlm, OpenAIConfiguration

llm = OpenAILlm(
    api_key=api_key,
    api_base=api_base,
    timeout=120,
    configuration=OpenAIConfiguration(
        model=model_name,
        temperature=0.0,
        max_tokens=16384,
    ),
)

In [3]:
from bridgic.core.agentic.tool_specs import FunctionToolSpec


async def research_topic(topic: str) -> str:
    """Search for information about a topic"""
    return f"Research results for '{topic}': Found 5 relevant sources with key insights."


async def create_outline(title: str, research: str) -> str:
    """Create an outline for an article"""
    return f"Outline for '{title}': Introduction, 3 main sections, conclusion"


async def write_section(title: str, section: str, notes: str) -> str:
    """Write an article section based on outline and research"""
    return f"Written section '{section}' for '{title}' ({len(notes)} chars of notes used)"


async def edit_article(title: str, edit_instructions: str) -> str:
    """Edit an existing article to fix issues"""
    return f"Edited '{title}': Applied changes — {edit_instructions}"


research_topic_tool = FunctionToolSpec.from_raw(research_topic)
create_outline_tool = FunctionToolSpec.from_raw(create_outline)
write_section_tool = FunctionToolSpec.from_raw(write_section)
edit_article_tool = FunctionToolSpec.from_raw(edit_article)

## Snapshot — Scoped Context Override

`async with self.snapshot(goal="...", **fields)` is the fundamental context scoping mechanism in Phase Annotation.

Here's what it does:

1. **On enter** — saves the current context state, then applies your overrides (goal, or any other context field)
2. **During execution** — all think units inside the block see the overridden context. The LLM receives the snapshot's goal, not the original one
3. **On exit** — restores all original field values, as if the snapshot block never happened

Snapshot also manages `LayeredExposure` state (skills, cognitive_history): it **clears** any previously revealed details on enter, so the LLM starts fresh in each phase. On exit, the pre-snapshot revealed state is restored.

This makes snapshot ideal for:
- Breaking a large task into focused sub-goals
- Handling interruptions without losing the outer context
- Scoping what the LLM "knows" in each phase

### Snapshot Lifecycle

```  
async with self.snapshot(goal="Sub-task A"):
                    ├─ save original fields
                    ├─ apply overrides (goal → "Sub-task A")
                    ├─ clear LayeredExposure revealed state
                    │
            await self.worker   ← LLM sees goal = "Sub-task A"
                    │
                    └─ restore original fields + revealed state

# Here, context.goal is back to the original
```

### Example: Multi-Phase Content Creation

Let's build an agent that works in two scoped phases — research, then writing. Each phase has its own goal, so the LLM focuses on one thing at a time.

In [ ]:
from bridgic.amphibious import (
    AmphibiousAutoma, CognitiveContext, CognitiveWorker, think_unit
)


class ContentCreator(AmphibiousAutoma[CognitiveContext]):
    researcher = think_unit(
        CognitiveWorker.inline(
            "Research the topic thoroughly. Gather key facts and sources.",
            verbose_prompt=True,
        ),
        max_attempts=3,
    )
    writer = think_unit(
        CognitiveWorker.inline(
            "Write the article based on the research gathered so far.",
            verbose_prompt=True,
        ),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        # Phase 1: Research — LLM sees goal = "Gather research material..."
        async with self.snapshot(goal="Gather research material on renewable energy"):
            await self.researcher

        # Phase 2: Write — LLM sees goal = "Write the article..."
        async with self.snapshot(goal="Write the article using the gathered research"):
            await self.writer

In [5]:
agent = ContentCreator(llm=llm, verbose=True)
result = await agent.arun(
    goal="Write an article about the future of renewable energy",
    tools=[research_topic_tool, create_outline_tool, write_section_tool],
)
print(result)

[18:36:34.976] [Router] (_amphibious_automa.py:1573) Auto-detecting execution mode
[18:36:34.976] [Router] (_amphibious_automa.py:1579) Detected AGENT mode
[18:36:34.977] [Observe] (_amphibious_automa.py:861) _PromptWorker: None
[18:36:40.473] [Think] (_cognitive_worker.py:332) Message 1 (system, 348 tokens):
Research the topic thoroughly. Gather key facts and sources.Respond in JSON format.

# Available Tools (with parameters):
• research_topic: Search for information about a topic
  - topic (string) [required]
• create_outline: Create an outline for an article
  - title (string) [required]
  - research (string) [required]
• write_section: Write an article section based on outline and research
  - title (string) [required]
  - section (string) [required]
  - notes (string) [required]

# Context Acquiring
If the context contains progressively disclosed information (e.g. skills, history steps) and you want to inspect the details, use the **details** field to request them. The framework 

In the verbose output above, notice how the LLM's goal changes between phases:

- During the first snapshot, it sees **"Gather research material on renewable energy"** and focuses on calling `research_topic`
- During the second snapshot, it sees **"Write the article using the gathered research"** and focuses on calling `write_section`

Each snapshot scopes the LLM's focus to one sub-task. The agent's overall goal ("Write an article about the future of renewable energy") is temporarily replaced inside each block, then restored on exit.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored Phase Annotation — structured context scoping for agent execution phases:

- **`self.snapshot(goal="...", **fields)`** is the core mechanism. It creates a scoped context where think units run, then automatically restores the original state on exit.
  - On enter: saves original fields, applies overrides, clears `LayeredExposure` revealed state
  - On exit: restores all fields and revealed state
- Snapshot is ideal for **sub-goal scoping**, **interruption handling**, and **context isolation** between phases.